In [2]:
import numpy as np 
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import KFold

In [3]:
X_train = pd.read_csv('../data/X_train.csv',index_col='ROW_ID')

y_train = pd.read_csv('../data/y_train.csv',index_col='ROW_ID')
total_data = pd.concat([X_train,y_train],axis=1)
print(total_data['target'])

ROW_ID
0         0.009210
1         0.000405
2         0.005967
3        -0.000542
4        -0.002579
            ...   
527068   -0.001769
527069    0.005640
527070    0.004016
527071    0.001617
527072    0.000556
Name: target, Length: 527073, dtype: float64


Time to look at the returns and how they interact with the target

In [4]:
# sign-based autocorrelation: does yesterday's direction predict tomorrow's?
ret_cols = [f'RET_{i}' for i in range(1, 21)]

target_sign = np.sign(total_data['target'])

for col in ret_cols:
    ret_sign = np.sign(total_data[col])
    # simple match rate: how often does lag-i's sign match tomorrow's sign?
    match_rate = (ret_sign == target_sign).mean() * 100
    print(f"{col}: sign-match rate with target = {match_rate:.2f}%")

RET_1: sign-match rate with target = 51.89%
RET_2: sign-match rate with target = 49.75%
RET_3: sign-match rate with target = 49.95%
RET_4: sign-match rate with target = 49.73%
RET_5: sign-match rate with target = 49.96%
RET_6: sign-match rate with target = 50.16%
RET_7: sign-match rate with target = 50.40%
RET_8: sign-match rate with target = 50.45%
RET_9: sign-match rate with target = 50.43%
RET_10: sign-match rate with target = 50.21%
RET_11: sign-match rate with target = 50.28%
RET_12: sign-match rate with target = 50.38%
RET_13: sign-match rate with target = 50.18%
RET_14: sign-match rate with target = 50.25%
RET_15: sign-match rate with target = 50.12%
RET_16: sign-match rate with target = 50.17%
RET_17: sign-match rate with target = 49.94%
RET_18: sign-match rate with target = 50.23%
RET_19: sign-match rate with target = 50.29%
RET_20: sign-match rate with target = 50.30%


In [5]:
for col in ret_cols:
    corr = total_data[col].corr(total_data['target'])
    print(f"{col}: Pearson corr with target = {corr:.4f}")

RET_1: Pearson corr with target = 0.0849
RET_2: Pearson corr with target = -0.0090
RET_3: Pearson corr with target = -0.0121
RET_4: Pearson corr with target = -0.0243
RET_5: Pearson corr with target = -0.0014
RET_6: Pearson corr with target = 0.0118
RET_7: Pearson corr with target = 0.0212
RET_8: Pearson corr with target = 0.0207
RET_9: Pearson corr with target = 0.0196
RET_10: Pearson corr with target = -0.0020
RET_11: Pearson corr with target = 0.0025
RET_12: Pearson corr with target = -0.0016
RET_13: Pearson corr with target = 0.0048
RET_14: Pearson corr with target = 0.0076
RET_15: Pearson corr with target = 0.0021
RET_16: Pearson corr with target = 0.0103
RET_17: Pearson corr with target = -0.0030
RET_18: Pearson corr with target = -0.0100
RET_19: Pearson corr with target = 0.0044
RET_20: Pearson corr with target = 0.0027


So the strongest signal is yesterdays returns, with the rest having limited corilation. its strong but not masivly worth checking if they alone can perform well 

Testing signed volumes i ran in to an issue with 73% of signed 1 being nul this make comparison difficult adding noise to the model meaning i have to skip considering the returns only the day before was meaningful

In [6]:
volume_cols = [f'SIGNED_VOLUME_{i}' for i in range(1, 21)]

null_summary = total_data[volume_cols].isna().sum()
null_pct = (null_summary / len(total_data) * 100).round(2)

print(pd.DataFrame({'null_count': null_summary, 'null_pct': null_pct}))

                  null_count  null_pct
SIGNED_VOLUME_1       387506     73.52
SIGNED_VOLUME_2            0      0.00
SIGNED_VOLUME_3            0      0.00
SIGNED_VOLUME_4            0      0.00
SIGNED_VOLUME_5            0      0.00
SIGNED_VOLUME_6            0      0.00
SIGNED_VOLUME_7            2      0.00
SIGNED_VOLUME_8            4      0.00
SIGNED_VOLUME_9            6      0.00
SIGNED_VOLUME_10           9      0.00
SIGNED_VOLUME_11          13      0.00
SIGNED_VOLUME_12          18      0.00
SIGNED_VOLUME_13          26      0.00
SIGNED_VOLUME_14          34      0.01
SIGNED_VOLUME_15          42      0.01
SIGNED_VOLUME_16          50      0.01
SIGNED_VOLUME_17          58      0.01
SIGNED_VOLUME_18         859      0.16
SIGNED_VOLUME_19        3002      0.57
SIGNED_VOLUME_20        8498      1.61


In [7]:

ret1_sign = np.sign(total_data['RET_1'])
target_sign = np.sign(total_data['target'])

match_rate = (ret1_sign == target_sign).mean() * 100
print(f"RET_1 sign-match rate with target = {match_rate:.2f}%")

# also check overall predicted-positive rate if you naively used sign(RET_1) as the prediction
pred_positive_rate = (ret1_sign == 1).mean() * 100
print(f"sign(RET_1) predicted-positive rate = {pred_positive_rate:.2f}%")

RET_1 sign-match rate with target = 51.89%
sign(RET_1) predicted-positive rate = 51.17%


Lets test to see if the daliy turn over match up similarly

In [8]:
# 1. data quality check first, given the SIGNED_VOLUME_1 surprise
print(total_data['MEDIAN_DAILY_TURNOVER'].describe())
print(f"\nNaN count: {total_data['MEDIAN_DAILY_TURNOVER'].isna().sum()}")
print(f"Zero count: {(total_data['MEDIAN_DAILY_TURNOVER'] == 0).sum()}")

# 2. correlation with target (magnitude, not sign — turnover has no direction)
corr = total_data['MEDIAN_DAILY_TURNOVER'].corr(total_data['target'])
print(f"\nMEDIAN_DAILY_TURNOVER: Pearson corr with target = {corr:.4f}")

# 3. correlation with the *magnitude* of target (does high turnover predict bigger moves, regardless of direction?)
abs_corr = total_data['MEDIAN_DAILY_TURNOVER'].corr(total_data['target'].abs())
print(f"MEDIAN_DAILY_TURNOVER: Pearson corr with |target| = {abs_corr:.4f}")

# 4. does turnover correlate with RET_1's *predictive power*? i.e. is RET_1 more reliable
#    when turnover is high vs low (a plausible hypothesis: high-turnover allocations may have
#    fresher/more meaningful RET_1 signal)
median_turnover = total_data['MEDIAN_DAILY_TURNOVER'].median()
high_turnover = total_data['MEDIAN_DAILY_TURNOVER'] > median_turnover

ret1_sign = np.sign(total_data['RET_1'])
target_sign = np.sign(total_data['target'])

match_high = (ret1_sign[high_turnover] == target_sign[high_turnover]).mean() * 100
match_low = (ret1_sign[~high_turnover] == target_sign[~high_turnover]).mean() * 100
print(f"\nRET_1 sign-match rate when turnover ABOVE median: {match_high:.2f}%")
print(f"RET_1 sign-match rate when turnover BELOW median: {match_low:.2f}%")


count    5.234350e+05
mean     9.094372e-02
std      2.364887e-01
min      6.917210e-16
25%      1.892675e-03
50%      1.593524e-02
75%      4.449061e-02
max      1.560090e+00
Name: MEDIAN_DAILY_TURNOVER, dtype: float64

NaN count: 3638
Zero count: 0

MEDIAN_DAILY_TURNOVER: Pearson corr with target = 0.0134
MEDIAN_DAILY_TURNOVER: Pearson corr with |target| = 0.0352

RET_1 sign-match rate when turnover ABOVE median: 52.09%
RET_1 sign-match rate when turnover BELOW median: 51.70%


RET_1 carries real short-horizon momentum signal (correlation ~7x stronger than any other lag), and MEDIAN_DAILY_TURNOVER modulates its reliability (52.09% vs 51.70% match rate above/below median turnover) — together these justify using just these two features rather than the full 20-lag history."